# 07 - Limpieza bibliográfica final

Esta versión reconstruye los campos bibliográficos desde `Canonico_Union_Trabajo.csv` usando `Base_origen + indice`, conserva autores/afiliaciones ya normalizados y deja exactamente 16 columnas.

No deduplica publicaciones y no modifica `Area` ni `SubArea`.

In [1]:
import os
import re
import html
import unicodedata
import pandas as pd

archivo_entrada = "../04_Limpieza/02_normalizacion/autores_unam_normalizados.csv"
archivo_canonico = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"

carpeta_salida = "../04_Limpieza/03_limpieza_bibliografica"
archivo_salida = f"{carpeta_salida}/autores_unam_limpios.csv"

os.makedirs(carpeta_salida, exist_ok=True)

columnas = [
    "Base_origen", "Fuente_origen", "indice", "Titulo", "Año",
    "Autor_norm", "Afiliacion1", "Afiliacion2", "ISBN", "ISSN",
    "Doi", "URL", "Area", "SubArea", "Keywords", "Abstract"
]

clave = ["Base_origen", "indice"]


In [2]:
# Funciones sencillas de limpieza

def nfc(texto):
    return unicodedata.normalize("NFC", texto or "")


def decodificar_html(texto):
    texto = texto or ""
    for _ in range(3):
        nuevo = html.unescape(texto)
        if nuevo == texto:
            break
        texto = nuevo
    return nfc(texto)


def limpiar_titulo(texto):
    texto = decodificar_html(texto)
    return re.sub(r"\s+", " ", texto).strip()


def limpiar_anio(texto):
    texto = (texto or "").strip()
    m = re.fullmatch(r"(\d{4})(?:\.0)?", texto)
    return m.group(1) if m else texto


def isbn13_valido(isbn):
    if not re.fullmatch(r"\d{13}", isbn):
        return False
    total = sum((1 if i % 2 == 0 else 3) * int(c) for i, c in enumerate(isbn[:12]))
    return (10 - total % 10) % 10 == int(isbn[-1])


def isbn10_valido(isbn):
    isbn = isbn.upper()
    if not re.fullmatch(r"\d{9}[\dX]", isbn):
        return False
    total = 0
    for i, c in enumerate(isbn):
        valor = 10 if c == "X" else int(c)
        total += (10 - i) * valor
    return total % 11 == 0


def limpiar_isbn(texto):
    texto = (texto or "").strip()
    if not texto:
        return ""

    # Nunca reconstruir una notación científica.
    if re.search(r"(?i)e[+-]?\d+", texto):
        return ""

    resultado = []

    for parte in re.split(r";|\|", texto):
        parte = parte.strip()
        if not parte:
            continue

        # Recupera ISBN-13 incluso cuando viene seguido por /24/07, /2025/06, etc.
        encontrados = re.findall(r"97[89](?:[-\s]?\d){10}", parte)

        if encontrados:
            candidatos = [re.sub(r"[^0-9]", "", x) for x in encontrados]
        else:
            candidatos = [re.sub(r"[^0-9Xx]", "", parte).upper()]

        for isbn in candidatos:
            valido = (
                (len(isbn) == 13 and isbn13_valido(isbn))
                or (len(isbn) == 10 and isbn10_valido(isbn))
            )
            if valido and isbn not in resultado:
                resultado.append(isbn)

    return "; ".join(resultado)


def issn_valido(issn):
    limpio = issn.replace("-", "").upper()
    if not re.fullmatch(r"\d{7}[\dX]", limpio):
        return False

    total = sum((8 - i) * int(limpio[i]) for i in range(7))
    control = (11 - total % 11) % 11
    esperado = "X" if control == 10 else str(control)
    return limpio[-1] == esperado


def limpiar_issn(texto):
    texto = (texto or "").strip()
    if not texto:
        return ""

    resultado = []

    for parte in re.split(r";|\|", texto):
        limpio = re.sub(r"[^0-9Xx]", "", parte).upper()

        if len(limpio) == 8:
            issn = limpio[:4] + "-" + limpio[4:]
            if issn_valido(issn) and issn not in resultado:
                resultado.append(issn)

    return "; ".join(resultado)


def limpiar_doi(texto):
    texto = decodificar_html(texto).strip()
    texto = re.sub(r"(?i)^https?://(?:dx\.)?doi\.org/", "", texto)
    texto = re.sub(r"(?i)^doi:\s*", "", texto)
    return texto.strip().lower()


def limpiar_url(texto):
    return decodificar_html(texto).strip()


def quitar_etiquetas_html(texto):
    texto = re.sub(r"(?i)<\s*br\s*/?\s*>", " ", texto)
    texto = re.sub(r"(?i)</?\s*(?:p|div|li)\b[^>]*>", " ", texto)
    return re.sub(r"<[^>]+>", "", texto)


def limpiar_keywords(texto):
    texto = quitar_etiquetas_html(decodificar_html(texto))

    # Quita URLs sin comerse las keywords que vienen después de un punto y coma.
    texto = re.sub(r"(?i)https?://[^;\s]+", " ", texto)

    resultado = []
    for parte in texto.split(";"):
        parte = re.sub(r"\s+", " ", parte).strip(" ,\t\r\n")
        if not parte:
            continue

        if re.search(r"(?i)\b(?:arnumber|isnumber|querytext|refinements?|tag)\s*=", parte):
            continue

        if parte not in resultado:
            resultado.append(parte)

    return "; ".join(resultado)


def limpiar_abstract(texto):
    texto = quitar_etiquetas_html(decodificar_html(texto))
    return re.sub(r"\s+", " ", texto).strip()


In [3]:
# Cargar archivos
entrada = pd.read_csv(
    archivo_entrada,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

canonico = pd.read_csv(
    archivo_canonico,
    dtype=str,
    keep_default_na=False,
    encoding="utf-8-sig"
)

print("Entrada física:", entrada.shape)
print("Canónico:", canonico.shape)

# Elimina las cuatro columnas Unnamed que arrastra la entrada.
faltantes = [c for c in columnas if c not in entrada.columns]
if faltantes:
    raise ValueError(f"Faltan columnas canónicas: {faltantes}")

entrada = entrada[columnas].copy()

if canonico.duplicated(clave).any():
    raise ValueError("Canonico_Union_Trabajo tiene claves Base_origen + indice duplicadas.")

claves_entrada = set(map(tuple, entrada[clave].drop_duplicates().to_numpy()))
claves_canonico = set(map(tuple, canonico[clave].to_numpy()))

faltan_claves = claves_entrada - claves_canonico
if faltan_claves:
    raise ValueError(f"Hay {len(faltan_claves)} publicaciones sin referencia en Canonico_Union_Trabajo.")

print("Entrada canónica:", entrada.shape)
print("Publicaciones de trabajo:", len(claves_entrada))
print("Cobertura en Canonico_Union_Trabajo: OK")


Entrada física: (4266, 20)
Canónico: (2153, 16)
Entrada canónica: (4266, 16)
Publicaciones de trabajo: 2109
Cobertura en Canonico_Union_Trabajo: OK


In [4]:
# Preparar la referencia bibliográfica limpia por publicación
campos_bibliograficos = [
    "Titulo", "Año", "ISBN", "ISSN", "Doi", "URL", "Keywords", "Abstract"
]

referencia = canonico[clave + campos_bibliograficos].copy()

referencia["Titulo"] = referencia["Titulo"].map(limpiar_titulo)
referencia["Año"] = referencia["Año"].map(limpiar_anio)
referencia["ISBN"] = referencia["ISBN"].map(limpiar_isbn)
referencia["ISSN"] = referencia["ISSN"].map(limpiar_issn)
referencia["Doi"] = referencia["Doi"].map(limpiar_doi)
referencia["URL"] = referencia["URL"].map(limpiar_url)
referencia["Keywords"] = referencia["Keywords"].map(limpiar_keywords)
referencia["Abstract"] = referencia["Abstract"].map(limpiar_abstract)

# Renombrar temporalmente para unir sin pisar todavía la entrada.
referencia = referencia.rename(
    columns={c: f"{c}_ref" for c in campos_bibliograficos}
)

trabajo = entrada.merge(
    referencia,
    on=clave,
    how="left",
    validate="many_to_one"
)

salida = entrada.copy()

for campo in campos_bibliograficos:
    salida[campo] = trabajo[f"{campo}_ref"].values

# Area y SubArea se conservan exactamente como estaban.
salida = salida[columnas].copy()


In [5]:
# Validaciones finales
assert len(salida) == len(entrada), "Cambió el número de filas."
assert list(salida.columns) == columnas, "La salida no tiene exactamente las 16 columnas canónicas."

for campo in [
    "Base_origen", "Fuente_origen", "indice",
    "Autor_norm", "Afiliacion1", "Afiliacion2",
    "Area", "SubArea"
]:
    assert entrada[campo].equals(salida[campo]), f"Se modificó {campo}."

assert salida["SubArea"].str.strip().eq("").all(), "SubArea dejó de estar vacía."

isbn_cientifico = salida["ISBN"].str.contains(r"(?i)e[+-]?\d+", regex=True).sum()
assert isbn_cientifico == 0, f"Quedan {isbn_cientifico} ISBN en notación científica."

for valor in salida["ISBN"]:
    if not valor:
        continue
    for isbn in valor.split("; "):
        assert isbn13_valido(isbn) or isbn10_valido(isbn), f"ISBN inválido: {isbn}"

for valor in salida["ISSN"]:
    if not valor:
        continue
    for issn in valor.split("; "):
        assert re.fullmatch(r"\d{4}-[\dX]{4}", issn), f"ISSN mal formado: {issn}"
        assert issn_valido(issn), f"ISSN inválido: {issn}"

assert salida["Año"].map(lambda x: x == "" or bool(re.fullmatch(r"\d{4}", x))).all()
assert salida["Doi"].map(lambda x: x == "" or x.startswith("10.")).all()
assert not salida["Doi"].str.contains(r"(?i)doi\.org", regex=True).any()
assert not salida["Keywords"].str.contains(
    r"(?i)https?://|arnumber=|isnumber=|querytext=", regex=True
).any()

# Todos los autores de la misma publicación deben compartir la misma bibliografía.
for campo in [
    "Titulo", "Año", "ISBN", "ISSN", "Doi", "URL",
    "Area", "SubArea", "Keywords", "Abstract"
]:
    conflictos = salida.groupby(clave)[campo].nunique(dropna=False).gt(1).sum()
    assert conflictos == 0, f"Hay {conflictos} publicaciones con conflicto en {campo}."

print("VALIDACIONES COMPLETADAS CORRECTAMENTE")
print("Filas finales:", len(salida))
print("Columnas finales:", len(salida.columns))
print("ISBN en notación científica:", isbn_cientifico)
print("ISSN no vacíos:", salida["ISSN"].str.strip().ne("").sum())


VALIDACIONES COMPLETADAS CORRECTAMENTE
Filas finales: 4266
Columnas finales: 16
ISBN en notación científica: 0
ISSN no vacíos: 3276


In [6]:
# Guardar archivo limpio
salida.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo generado:", archivo_salida)


Archivo generado: ../04_Limpieza/03_limpieza_bibliografica/autores_unam_limpios.csv
